In [1]:
import json
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
from collections import Counter

In [2]:
# =========================
# 1) Cấu hình đường dẫn
# =========================
PATH_THANH  = Path("./Thanh.json")
PATH_TRANG  = Path("./Trang.json")
PATH_TRUONG = Path("./Truong.json")

In [3]:
CATEGORIES  = ["positive", "negative", "neutral"]
N_RATERS    = 3

In [4]:
# =========================
# 1) Hàm tiện ích
# =========================
def load_json_flex(path: Path):
    """Đọc JSON: hỗ trợ list-of-dicts hoặc JSON Lines."""
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = [data]
        return data
    except json.JSONDecodeError:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

def normalize_label(x: str):
    """Chuẩn hoá nhãn về {positive, negative, neutral}."""
    if x is None:
        return None
    s = str(x).strip().lower()
    mapping = {
        "pos": "positive", "positive": "positive", "+": "positive",
        "neg": "negative", "negative": "negative", "-": "negative",
        "neu": "neutral",  "neutral": "neutral",  "0": "neutral"
    }
    return mapping.get(s, None)

def canonicalize_text(t: str):
    """Chuẩn hoá text: NFC, strip, thay nhiều khoảng trắng bằng 1 khoảng."""
    if t is None:
        return None
    # Chuẩn hoá Unicode NFC để so khớp bền vững dấu tiếng Việt
    t = unicodedata.normalize("NFC", str(t))
    # Thay các xuống dòng/tab bằng khoảng trắng và nén khoảng trắng
    t = " ".join(t.split())
    return t

def detect_text_key(rows):
    """Đoán cột text phổ biến."""
    if not rows:
        return None
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    # Ưu tiên 'text', sau đó một số khoá thường gặp
    for k in ["text", "content", "body", "sentence", "review"]:
        if k in keys:
            return k
    # Thử ghép title + content nếu có
    if "title" in keys and "content" in keys:
        return ("title", "content")
    return None

def to_df(rows, annotator_name: str):
    """Chuyển list[dict] -> DataFrame với cột: text_raw, text_canon, label."""
    if len(rows) == 0:
        return pd.DataFrame(columns=["text_raw", "text_canon", annotator_name])

    text_key = detect_text_key(rows)
    # Xác định cột nhãn có thể có
    candidate_label_keys = ["label", "sentiment", "y", "tag", "prediction"]

    # Suy ra label_key
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    label_key = next((k for k in candidate_label_keys if k in keys), None)
    if label_key is None:
        # Không tìm thấy cột nhãn -> None
        label_key = None

    df = pd.DataFrame(rows)

    # Lấy text_raw
    if isinstance(text_key, tuple):
        t1, t2 = text_key
        df["text_raw"] = (df.get(t1, "").astype(str) + " " + df.get(t2, "").astype(str)).str.strip()
    elif text_key is None:
        # Không xác định được text -> dùng repr của dòng (không khuyến nghị)
        df["text_raw"] = df.astype(str).agg(" ".join, axis=1)
    else:
        df["text_raw"] = df[text_key].astype(str)

    df["text_canon"] = df["text_raw"].apply(canonicalize_text)

    # Lấy nhãn
    if label_key is None:
        df[annotator_name] = None
    else:
        df[annotator_name] = df[label_key].apply(normalize_label)

    return df[["text_raw", "text_canon", annotator_name]]

def pick_mode_or_nan(series):
    """Chọn mode của một Series; nếu hòa (nhiều mode) -> NaN."""
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return np.nan
    cnt = Counter(vals)
    most_common = cnt.most_common()
    if len(most_common) == 1:
        return most_common[0][0]
    # kiểm tra hoà
    if len(most_common) >= 2 and most_common[0][1] == most_common[1][1]:
        return np.nan
    return most_common[0][0]

def fleiss_kappa(counts: np.ndarray) -> float:
    """
    counts: (n_items, k) với counts[i, j] = số người gán cho item i ở lớp j.
    """
    n, k = counts.shape
    N = np.sum(counts[0])  # số annotator mỗi item (phải bằng nhau)
    p_j = counts.sum(axis=0) / (n * N)                         # tỉ lệ theo lớp
    P_i = (np.sum(counts**2, axis=1) - N) / (N * (N - 1))      # đồng thuận từng item
    P_bar = np.mean(P_i)
    P_e = np.sum(p_j**2)
    if np.isclose(1 - P_e, 0):
        return np.nan
    return (P_bar - P_e) / (1 - P_e)

In [5]:
# =========================
# 2) Đọc & chuẩn hoá từng annotator
# =========================
rows_thanh  = load_json_flex(PATH_THANH)
rows_trang  = load_json_flex(PATH_TRANG)
rows_truong = load_json_flex(PATH_TRUONG)

df_thanh  = to_df(rows_thanh,  "Thanh")
df_trang  = to_df(rows_trang,  "Trang")
df_truong = to_df(rows_truong, "Truong")

In [6]:
df_trang

,text_raw,text_canon,Trang
0,Tiến sĩ Toán học hai lần đạt điểm tuyệt đối Ol...,Tiến sĩ Toán học hai lần đạt điểm tuyệt đối Ol...,positive
1,Ai là vị vua được nhân dân gọi là 'Vua Đầm Đêm...,Ai là vị vua được nhân dân gọi là 'Vua Đầm Đêm...,positive
2,Phó giáo sư bị gỡ bài báo đứng tên cùng học si...,Phó giáo sư bị gỡ bài báo đứng tên cùng học si...,negative
3,Gia cảnh khốn khó của nữ sinh lớp 12 sau vụ ta...,Gia cảnh khốn khó của nữ sinh lớp 12 sau vụ ta...,negative
4,Trao hơn 52 triệu đồng của bạn đọc ủng hộ gia ...,Trao hơn 52 triệu đồng của bạn đọc ủng hộ gia ...,negative
...,...,...,...
195,Eurowindow Holding sắp khởi công dự án khu đô ...,Eurowindow Holding sắp khởi công dự án khu đô ...,neutral
196,Chính phủ yêu cầu bỏ ngay quy chuẩn mâu thuẫn ...,Chính phủ yêu cầu bỏ ngay quy chuẩn mâu thuẫn ...,neutral
197,Mua nhà tiêu chí mới: Chuẩn mực sống thưởng th...,Mua nhà tiêu chí mới: Chuẩn mực sống thưởng th...,neutral
198,"Tiền đất bổ sung: Lỗi không do doanh nghiệp, n...","Tiền đất bổ sung: Lỗi không do doanh nghiệp, n...",negative


In [7]:
df_thanh

,text_raw,text_canon,Thanh
0,Tiến sĩ Toán học hai lần đạt điểm tuyệt đối Ol...,Tiến sĩ Toán học hai lần đạt điểm tuyệt đối Ol...,positive
1,Ai là vị vua được nhân dân gọi là 'Vua Đầm Đêm...,Ai là vị vua được nhân dân gọi là 'Vua Đầm Đêm...,neutral
2,Phó giáo sư bị gỡ bài báo đứng tên cùng học si...,Phó giáo sư bị gỡ bài báo đứng tên cùng học si...,negative
3,Gia cảnh khốn khó của nữ sinh lớp 12 sau vụ ta...,Gia cảnh khốn khó của nữ sinh lớp 12 sau vụ ta...,negative
4,Trao hơn 52 triệu đồng của bạn đọc ủng hộ gia ...,Trao hơn 52 triệu đồng của bạn đọc ủng hộ gia ...,positive
...,...,...,...
195,Eurowindow Holding sắp khởi công dự án khu đô ...,Eurowindow Holding sắp khởi công dự án khu đô ...,positive
196,Chính phủ yêu cầu bỏ ngay quy chuẩn mâu thuẫn ...,Chính phủ yêu cầu bỏ ngay quy chuẩn mâu thuẫn ...,negative
197,Mua nhà tiêu chí mới: Chuẩn mực sống thưởng th...,Mua nhà tiêu chí mới: Chuẩn mực sống thưởng th...,positive
198,"Tiền đất bổ sung: Lỗi không do doanh nghiệp, n...","Tiền đất bổ sung: Lỗi không do doanh nghiệp, n...",negative


In [8]:
df_truong

,text_raw,text_canon,Truong
0,Tiến sĩ Toán học hai lần đạt điểm tuyệt đối Ol...,Tiến sĩ Toán học hai lần đạt điểm tuyệt đối Ol...,positive
1,Ai là vị vua được nhân dân gọi là 'Vua Đầm Đêm...,Ai là vị vua được nhân dân gọi là 'Vua Đầm Đêm...,positive
2,Phó giáo sư bị gỡ bài báo đứng tên cùng học si...,Phó giáo sư bị gỡ bài báo đứng tên cùng học si...,negative
3,Gia cảnh khốn khó của nữ sinh lớp 12 sau vụ ta...,Gia cảnh khốn khó của nữ sinh lớp 12 sau vụ ta...,negative
4,Trao hơn 52 triệu đồng của bạn đọc ủng hộ gia ...,Trao hơn 52 triệu đồng của bạn đọc ủng hộ gia ...,negative
...,...,...,...
195,Eurowindow Holding sắp khởi công dự án khu đô ...,Eurowindow Holding sắp khởi công dự án khu đô ...,positive
196,Chính phủ yêu cầu bỏ ngay quy chuẩn mâu thuẫn ...,Chính phủ yêu cầu bỏ ngay quy chuẩn mâu thuẫn ...,negative
197,Mua nhà tiêu chí mới: Chuẩn mực sống thưởng th...,Mua nhà tiêu chí mới: Chuẩn mực sống thưởng th...,positive
198,"Tiền đất bổ sung: Lỗi không do doanh nghiệp, n...","Tiền đất bổ sung: Lỗi không do doanh nghiệp, n...",negative


In [9]:
# =========================
# 3) Gom theo text_canon trong từng file
#    - Nếu 1 annotator gán nhiều nhãn cho cùng 1 text: lấy mode; nếu hoà -> bỏ (NaN)
# =========================
df_thanh_agg = (df_thanh.groupby(["text_canon"], as_index=False)
                        .agg(text_raw=("text_raw", "first"),
                             Thanh=("Thanh", pick_mode_or_nan)))

df_trang_agg = (df_trang.groupby(["text_canon"], as_index=False)
                        .agg(text_raw=("text_raw", "first"),
                             Trang=("Trang", pick_mode_or_nan)))

df_truong_agg = (df_truong.groupby(["text_canon"], as_index=False)
                          .agg(text_raw=("text_raw", "first"),
                               Truong=("Truong", pick_mode_or_nan)))

In [10]:
# =========================
# 4) Lấy GIAO các text xuất hiện ở cả 3 file
# =========================
common = (df_thanh_agg.merge(df_trang_agg[["text_canon", "Trang"]], on="text_canon", how="inner")
                      .merge(df_truong_agg[["text_canon", "Truong"]], on="text_canon", how="inner"))

# Loại các dòng thiếu nhãn (NaN do hoà hoặc thiếu)
common = common.dropna(subset=["Thanh", "Trang", "Truong"])

print("Số text chung (sau khi lọc đầy đủ 3 nhãn & hoà loại bỏ):", len(common))


Số text chung (sau khi lọc đầy đủ 3 nhãn & hoà loại bỏ): 200


In [11]:
# =========================
# 5) Tạo ma trận đếm cho Fleiss
# =========================
def row_to_counts(row):
    labels = [row["Thanh"], row["Trang"], row["Truong"]]
    return [sum(1 for lb in labels if lb == c) for c in CATEGORIES]

counts_matrix = np.vstack(common.apply(row_to_counts, axis=1).values)

# Kiểm tra tổng phiếu từng dòng
assert np.all(counts_matrix.sum(axis=1) == N_RATERS), "Có dòng tổng phiếu != số annotator!"

In [12]:
# =========================
# 6) Tính Fleiss’s Kappa
# =========================
kappa = fleiss_kappa(counts_matrix)

# =========================
# 7) Báo cáo nhanh
# =========================
print("Fleiss' Kappa (3 annotators, 3 lớp) trên GIAO text:", round(kappa, 4))

# Phân bố đa số để tham khảo
majority_label = [CATEGORIES[np.argmax(row)] for row in counts_matrix]
majority_counts = pd.Series(majority_label).value_counts().reindex(CATEGORIES, fill_value=0)
print("\nPhân bố nhãn theo đa số (trên tập giao):")
print(majority_counts.to_string())

# Tỷ lệ phiếu theo lớp (p_j)
p_j = counts_matrix.sum(axis=0) / counts_matrix.sum()
print("\nTỷ lệ phiếu theo lớp (p_j):")
for cls, pj in zip(CATEGORIES, p_j):
    print(f"  {cls:>8}: {pj:.3f}")

# Nếu cần xuất kết quả chi tiết để kiểm tra:
common.to_csv("./common_texts_with_labels.csv", index=False, encoding="utf-8-sig")
print("Đã lưu chi tiết:", "./common_texts_with_labels.csv")

Fleiss' Kappa (3 annotators, 3 lớp) trên GIAO text: 0.6671

Phân bố nhãn theo đa số (trên tập giao):
positive    68
negative    67
neutral     65

Tỷ lệ phiếu theo lớp (p_j):
  positive: 0.312
  negative: 0.342
   neutral: 0.347
Đã lưu chi tiết: ./common_texts_with_labels.csv


# Trường hợp bất đồng nhau


In [13]:
# Find cases where annotators disagree on labels, and save them to a CSV.
# - Align items by canonicalized text intersection across the three files.
# - Normalize labels to {positive, negative, neutral}.
# - Output rows where at least one annotator differs.
# - Save to ./label_disagreements.csv

import json, unicodedata
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

PATH_THANH  = Path("./Thanh.json")
PATH_TRANG  = Path("./Trang.json")
PATH_TRUONG = Path("./Truong.json")

CATEGORIES = ["positive", "negative", "neutral"]

def load_json_flex(path: Path):
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = [data]
        return data
    except json.JSONDecodeError:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

def detect_text_key(rows):
    if not rows:
        return None
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    for k in ["text", "content", "body", "sentence", "review"]:
        if k in keys:
            return k
    if "title" in keys and "content" in keys:
        return ("title", "content")
    return None

def canonicalize_text(t: str):
    if t is None:
        return None
    t = unicodedata.normalize("NFC", str(t))
    t = " ".join(t.split())
    return t

def normalize_label(x: str):
    if x is None:
        return None
    s = str(x).strip().lower()
    mapping = {
        "pos": "positive", "positive": "positive", "+": "positive",
        "neg": "negative", "negative": "negative", "-": "negative",
        "neu": "neutral",  "neutral": "neutral",  "0": "neutral"
    }
    return mapping.get(s, None)

def to_df(rows, annotator_name: str):
    if len(rows) == 0:
        return pd.DataFrame(columns=["text_raw", "text_canon", annotator_name])
    text_key = detect_text_key(rows)
    candidate_label_keys = ["label", "sentiment", "y", "tag", "prediction"]
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    label_key = next((k for k in candidate_label_keys if k in keys), None)
    df = pd.DataFrame(rows)
    if isinstance(text_key, tuple):
        t1, t2 = text_key
        df["text_raw"] = (df.get(t1, "").astype(str) + " " + df.get(t2, "").astype(str)).str.strip()
    elif text_key is None:
        df["text_raw"] = df.astype(str).agg(" ".join, axis=1)
    else:
        df["text_raw"] = df[text_key].astype(str)
    df["text_canon"] = df["text_raw"].apply(canonicalize_text)
    if label_key is None:
        df[annotator_name] = None
    else:
        df[annotator_name] = df[label_key].apply(normalize_label)
    return df[["text_raw", "text_canon", annotator_name]]

# Load and convert
rows_thanh  = load_json_flex(PATH_THANH)
rows_trang  = load_json_flex(PATH_TRANG)
rows_truong = load_json_flex(PATH_TRUONG)

df_thanh  = to_df(rows_thanh,  "Thanh")
df_trang  = to_df(rows_trang,  "Trang")
df_truong = to_df(rows_truong, "Truong")

# Aggregate by text_canon for each annotator (if duplicates, pick the most common label; ties -> NaN)
def pick_mode_or_nan(series):
    vals = [v for v in series if pd.notna(v)]
    if not vals:
        return np.nan
    c = Counter(vals).most_common()
    if len(c) == 1: 
        return c[0][0]
    if len(c) >= 2 and c[0][1] == c[1][1]:
        return np.nan
    return c[0][0]

agg_thanh  = df_thanh.groupby("text_canon", as_index=False).agg(text_raw=("text_raw", "first"),
                                                                Thanh=("Thanh", pick_mode_or_nan))
agg_trang  = df_trang.groupby("text_canon", as_index=False).agg(text_raw=("text_raw", "first"),
                                                                Trang=("Trang", pick_mode_or_nan))
agg_truong = df_truong.groupby("text_canon", as_index=False).agg(text_raw=("text_raw", "first"),
                                                                 Truong=("Truong", pick_mode_or_nan))

# Intersect texts across all three
common = agg_thanh.merge(agg_trang[["text_canon", "Trang"]], on="text_canon", how="inner") \
                  .merge(agg_truong[["text_canon", "Truong"]], on="text_canon", how="inner")

# Remove rows with any missing label (including ties)
common = common.dropna(subset=["Thanh", "Trang", "Truong"])

# Keep only disagreements
mask_disagree = ~((common["Thanh"] == common["Trang"]) & (common["Trang"] == common["Truong"]))
disagreements = common.loc[mask_disagree].copy()

# Optional: add a quick summary
def majority_vote(row):
    labels = [row["Thanh"], row["Trang"], row["Truong"]]
    cnt = Counter(labels).most_common()
    if len(cnt) == 0:
        return None
    if len(cnt) > 1 and cnt[0][1] == cnt[1][1]:
        return "tie"
    return cnt[0][0]

disagreements["majority_vote"] = disagreements.apply(majority_vote, axis=1)

# Save to CSV
out_path = "./label_disagreements.csv"
disagreements.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"Số văn bản chung (đủ 3 nhãn): {len(common)}")
print(f"Số trường hợp bất đồng nhãn: {len(disagreements)}")
print("Đã lưu CSV:", out_path)

# Print a preview of the first 10 disagreements
print("\n=== Preview 10 disagreements ===")
cols_to_show = ["text_raw", "Thanh", "Trang", "Truong", "majority_vote"]
for i, row in disagreements.head(10).iterrows():
    print("---")
    print("text_raw:", (row["text_raw"][:300] + ("..." if len(row["text_raw"])>300 else "")))
    print("Thanh:", row["Thanh"], "| Trang:", row["Trang"], "| Truong:", row["Truong"], "| majority:", row["majority_vote"])


Số văn bản chung (đủ 3 nhãn): 200
Số trường hợp bất đồng nhãn: 66
Đã lưu CSV: ./label_disagreements.csv

=== Preview 10 disagreements ===
---
text_raw: 'Đu trend' ở biển Kỳ Co, cô gái gặp sự cố dở khóc dở cười, phải vội tìm bác sĩ — Khi quay video tại bãi biển Kỳ Co, Bình Định, Thanh Huyền - một nữ du khách tới từ Hà Nội đã bị cơn sóng mạnh ập tới, đẩy nước và cát trào lên mặt, tràn vào trong tai.
Thanh: negative | Trang: negative | Truong: positive | majority: negative
---
text_raw: 437 thí sinh vắng thi môn Toán, 1 thí sinh bị đình chỉ sau 3 môn thi — Sáng 8/6, Sở GD-ĐT tổ chức thông tin về kết quả tổ chức kỳ thi tuyển sinh vào lớp 10 công lập không chuyên Hà Nội năm học 2025-2026.
Thanh: negative | Trang: negative | Truong: neutral | majority: negative
---
text_raw: Ai là vị vua được nhân dân gọi là 'Vua Đầm Đêm'? — Ông là vị vua nổi danh với chiến thuật đánh úp trong đêm, khiến kẻ thù khiếp sợ giữa vùng đất hiểm trở.
Thanh: neutral | Trang: positive | Truong: positive | majority: p

# Fill lại các dòng gán nhãn bị conflict


In [14]:
# Overwrite the original 'sentiment' field in Truong.json using corrected labels from CSV,
# then export to ./final-round1.json

import json, unicodedata, traceback
from pathlib import Path
import pandas as pd

CSV_PATH          = Path("./label_disagreements_fixed.csv")  # corrected labels
INPUT_JSON_PATH   = Path("./Thanh.json")               # choose which annotator JSON to update
OUTPUT_JSON_PATH  = Path("./final-round2.json")

def canonicalize_text(t: str):
    if t is None:
        return None
    t = unicodedata.normalize("NFC", str(t))
    t = " ".join(t.split())
    return t

def normalize_label(x: str):
    if x is None:
        return None
    s = str(x).strip().lower()
    mapping = {
        "pos": "positive", "positive": "positive", "+": "positive",
        "neg": "negative", "negative": "negative", "-": "negative",
        "neu": "neutral",  "neutral": "neutral",  "0": "neutral"
    }
    return mapping.get(s, s)

def load_json_flex(path: Path):
    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = [data]
        return data
    except json.JSONDecodeError:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rows.append(json.loads(line))
        return rows

def detect_text_key_from_rows(rows):
    if not rows:
        return None
    keys = set()
    for r in rows:
        if isinstance(r, dict):
            keys.update(r.keys())
    for k in ["text", "content", "body", "sentence", "review"]:
        if k in keys:
            return k
    if "title" in keys and "content" in keys:
        return ("title", "content")
    return None

def extract_text(row, text_key):
    if isinstance(text_key, tuple):
        k1, k2 = text_key
        return f"{row.get(k1,'') or ''} {row.get(k2,'') or ''}".strip()
    elif text_key is None:
        return " ".join([f"{k}:{v}" for k,v in row.items()])
    else:
        return str(row.get(text_key, ""))

try:
    # Load CSV
    df = pd.read_csv(CSV_PATH)
    # Detect columns
    df_cols_lower = {c.lower(): c for c in df.columns}
    text_col = df_cols_lower.get("text_canon") or df_cols_lower.get("review") or df_cols_lower.get("text") \
               or df_cols_lower.get("content") or df_cols_lower.get("body") or df_cols_lower.get("sentence") \
               or df_cols_lower.get("text_raw")
    label_col = df_cols_lower.get("label_corrected") or df_cols_lower.get("corrected_label") \
                or df_cols_lower.get("label_final") or df_cols_lower.get("final_label") \
                or df_cols_lower.get("label_revised") or df_cols_lower.get("revised_label") \
                or df_cols_lower.get("gold_label") or df_cols_lower.get("gold") \
                or df_cols_lower.get("truth") or df_cols_lower.get("ground_truth") \
                or df_cols_lower.get("sentiment_final") or df_cols_lower.get("final_sentiment") \
                or df_cols_lower.get("sentiment_revised") or df_cols_lower.get("label") \
                or df_cols_lower.get("sentiment")
    if not text_col or not label_col:
        raise RuntimeError("CSV thiếu cột text hoặc nhãn đã hiệu chỉnh.")
    # Build mapping
    df["_text_key"] = df[text_col].apply(canonicalize_text)
    df["_label_norm"] = df[label_col].apply(normalize_label)
    csv_map = {}
    for _, r in df.iterrows():
        key = r["_text_key"]
        val = r["_label_norm"]
        if key and isinstance(key, str) and isinstance(val, str) and val.strip():
            csv_map[key] = val

    # Load JSON and apply overwrite to 'sentiment'
    records = load_json_flex(INPUT_JSON_PATH)
    text_key_json = detect_text_key_from_rows(records)

    updated = 0
    changed = 0
    for rec in records:
        txt = extract_text(rec, text_key_json)
        key = canonicalize_text(txt)
        if key in csv_map:
            new_label = csv_map[key]
            old_label = rec.get("sentiment", None)
            rec["sentiment"] = new_label  # overwrite as requested
            updated += 1
            if old_label != new_label:
                changed += 1

    with OUTPUT_JSON_PATH.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print("=== Overwrite Summary ===")
    print("CSV rows:", len(df))
    print("Unique text keys in CSV:", len(csv_map))
    print("JSON records:", len(records))
    print("Records matched & overwritten (sentiment):", updated)
    print("Records where label actually changed:", changed)
    print("Saved:", str(OUTPUT_JSON_PATH))

except Exception as e:
    print("ERROR:", type(e).__name__, str(e))
    print(traceback.format_exc())


ERROR: FileNotFoundError [Errno 2] No such file or directory: 'label_disagreements_fixed.csv'
Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_7776\2343709951.py", line 72, in <module>
    df = pd.read_csv(CSV_PATH)
         ^^^^^^^^^^^^^^^^^^^^^
  File "d:\AppDownload\Anaconda\envs\llms\Lib\site-packages\pandas\io\parsers\readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AppDownload\Anaconda\envs\llms\Lib\site-packages\pandas\io\parsers\readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AppDownload\Anaconda\envs\llms\Lib\site-packages\pandas\io\parsers\readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\AppDownload\Anaconda\envs\llms\Lib\site-packages\pandas\io\par